# Parte 1 — Recuperación de Información (RI)

Corpus: discursos presidenciales en español — `unified/presidentes_latam_es.parquet`, 4,531 documentos.

Un **único** índice sostiene los tres buscadores:

- **TF-IDF** (`w_global`) — matriz doc×término, ranking por coseno
- **BM25** — listas de postings, ranking probabilístico
- **Booleana** — listas de postings + conjuntos, coincidencia exacta (AND / OR / NOT / frases)

Artefactos en `ri_scripts/artifacts/`: `corpus.jsonl`, `postings.npz`, `w_global.npz`, `vocabulary.json`, `idf.npy`, `bm25_stats.npz`, `index_meta.json`.

Preproceso: minúsculas · plegado de acentos (ñ conservada) · stopwords ES · stemming Snowball · `min_df=2`.

In [23]:
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import scipy.sparse as sp

from ri_scripts import config
from ri_scripts import (artifacts_exist, load_index, search_tfidf, search_bm25,
                        search_boolean, parse_boolean, analyze, STOPWORDS)
from ri_scripts.build_all import main as build_all
from ri_scripts.corpus import iter_corpus, read_doc

pd.set_option("display.max_colwidth", 60)

SHORT = {
    "Andrés Manuel López Obrador": "AMLO",
    "Javier Milei": "Milei",
    "Nicolás Maduro": "Maduro",
    "Pedro Sánchez": "Sánchez",
    "Sebastián Piñera Echenique": "Piñera",
}


def info(pares, ancho=36):
    # pares clave/valor en líneas alineadas
    for clave, valor in pares.items():
        if isinstance(valor, (int, np.integer)):
            valor = f"{valor:,}".replace(",", " ")
        print(f"  {clave:<{ancho}}{valor}")


def show(results, titulo=None, ancho_frag=100):
    # una línea por resultado + fragmento
    if titulo:
        print(f"\n{titulo}")
    if not results:
        print("  sin resultados")
        return
    con_score = len({r["score"] for r in results}) > 1
    for r in results:
        hablante = SHORT.get(r["speaker"], r["speaker"])
        score = f"{r['score']:.3f}  " if con_score else ""
        print(f"  {r['rank']}. {hablante:<8}{r['date']}  {score}{r['title'][:48]}")
        print(f"     {r['snippet'][:ancho_frag]}")


if not artifacts_exist():
    build_all(force=False)      # ~40 s la primera vez

ix = load_index()
w_global = ix.w_global          # matriz TF-IDF global, reutilizable

print("índice cargado")
info({
    "documentos": ix.n_docs,
    "vocabulario (df >= 2)": ix.vocab_size,
    "stopwords": len(STOPWORDS),
})
print(f"  ejemplo de análisis: {' '.join(analyze('Los Estados Unidos y la inflación del año 2024'))}")

índice cargado
  documentos                          4 531
  vocabulario (df >= 2)               30 083
  stopwords                           317
  ejemplo de análisis: unid inflacion año 2024


## Corpus cargado

`doc_id` es la posición en el corpus. Los documentos de cada hablante ocupan un bloque contiguo, así que filtrar por presidente es un rango de `doc_id`.

In [24]:
filas = []
for speaker, (start, end) in ix.speaker_blocks.items():
    block = np.arange(start, end)
    filas.append({
        "docs": len(block),
        "desde": min(ix.dates[i] for i in block),
        "hasta": max(ix.dates[i] for i in block),
        "tokens": f"{int(ix.dl[block].sum()):,}".replace(",", " "),
        "tok/doc": round(float(ix.dl[block].mean())),
        "doc_id": f"{start}-{end - 1}",
    })
tabla = pd.DataFrame(filas, index=[SHORT[s] for s in ix.speaker_blocks])
display(tabla.sort_values("docs", ascending=False))

,docs,desde,hasta,tokens,tok/doc,doc_id
AMLO,2294,2018-12-02,2024-09-30,8 488 220,3700,0-2293
Sánchez,1136,2018-06-11,2026-04-29,1 476 379,1300,2511-3646
Piñera,884,2018-03-11,2022-03-05,612 638,693,3647-4530
Milei,187,2023-12-10,2026-04-29,349 106,1867,2294-2480
Maduro,30,2015-07-06,2023-01-24,53 875,1796,2481-2510


## Estructura del índice

`df` = en cuántos documentos aparece un término · `idf` = `ln((1+N)/(1+df)) + 1`

In [25]:
info({
    "documentos (N)": ix.n_docs,
    "vocabulario (V)": ix.vocab_size,
    "entradas w_global": int(w_global.nnz),
    "densidad": f"{w_global.nnz / (ix.n_docs * ix.vocab_size):.2%}",
    "términos distintos por doc": round(w_global.nnz / ix.n_docs),
    "idf mínimo / máximo": f"{ix.idf.min():.2f} / {ix.idf.max():.2f}",
    "tokens por doc (avgdl)": round(ix.avgdl),
})

print("\n  términos más frecuentes:")
for j in np.argsort(-ix.df)[:10]:
    print(f"    {ix.terms[j]:<14}df {int(ix.df[j]):>5}    idf {ix.idf[j]:>5.2f}")

  documentos (N)                      4 531
  vocabulario (V)                     30 083
  entradas w_global                   3 687 905
  densidad                            2.71%
  términos distintos por doc          814
  idf mínimo / máximo                 1.05 / 8.32
  tokens por doc (avgdl)              2 423

  términos más frecuentes:
    pais          df  4312    idf  1.05
    hac           df  4306    idf  1.05
    much          df  4211    idf  1.07
    graci         df  4113    idf  1.10
    tod           df  4071    idf  1.11
    buen          df  4063    idf  1.11
    pod           df  4062    idf  1.11
    ser           df  4047    idf  1.11
    gobiern       df  4032    idf  1.12
    import        df  4019    idf  1.12


## Búsqueda TF-IDF (coseno)

La consulta se proyecta al mismo espacio (mismo analizador + IDF global) y se normaliza. Como las filas de `w_global` son unitarias, `w_global @ q` ya es el coseno.

In [26]:
for q in ["inflación y salarios", "seguridad pública", "corrupción", "libertad económica"]:
    show(search_tfidf(ix, q, k=4), f"TF-IDF · {q}")


TF-IDF · inflación y salarios
  1. AMLO    2019-12-16  0.548  16.12.19 Versión estenográfica. Salario Mínimo 2
     …debía aumentar el salario mínimo siempre por encima de la inflación. Incluso para el caso de los tr
  2. Milei   2026-04-14  0.309  Discurso del Presidente Javier Milei en el AmCha
     …cosas que hace la política tradicional, y como odio a la inflación, y como el dato no me gustó y me
  3. AMLO    2022-04-28  0.298  28.04.22 Versión estenográfica. 125 Asamblea Gen
     …el descontrol de la economía mundial, el incremento de la inflación, es algo que tenemos que enfren
  4. Milei   2024-04-25  0.242  Palabras del Presidente de la Nación, Javier Mil
     …que hoy está tan presente en Argentina sobre el tema de la inflación, sobre la discusión del tipo d

TF-IDF · seguridad pública
  1. AMLO    2019-10-31  0.276  31.10.19 Versión estenográfica. 110 Asamblea Gen
     …se ha consolidado con el tiempo y es una garantía para la seguridad social en nuestro país. Por la 
  2. A

## Búsqueda BM25

Recorre solo las listas de postings consultadas. `k1` satura la frecuencia, `b` controla la normalización por longitud. Al filtrar por hablante se usa su longitud media.

In [27]:
show(search_bm25(ix, "inflación y salarios", k=4), "BM25 · inflación y salarios")


BM25 · inflación y salarios
  1. AMLO    2019-12-16  6.326  16.12.19 Versión estenográfica. Salario Mínimo 2
     …debía aumentar el salario mínimo siempre por encima de la inflación. Incluso para el caso de los tr
  2. AMLO    2022-05-14  6.073  14.05.22 Versión estenográfica. Jornadas de prod
     …estamos nosotros seguros que vamos a hacerle frente a la inflación. Es un desafío. Otro más, nos to
  3. AMLO    2022-04-28  6.010  28.04.22 Versión estenográfica. 125 Asamblea Gen
     …el descontrol de la economía mundial, el incremento de la inflación, es algo que tenemos que enfren
  4. AMLO    2022-08-28  5.996  28.08.22 Versión estenográfica. Encuentro con tr
     …ahora que se precipitó la crisis económica y sobre todo la inflación y el aumento en el precio de l


In [28]:
print("\nefecto de b sobre 'seguridad' (top 3):\n")
for b in [0.0, 0.5, 0.75, 1.0]:
    top3 = search_bm25(ix, "seguridad", k=3, b=b)
    linea = "   ".join(f"{SHORT[r['speaker']]:<8}{r['date']} {r['score']:.2f}" for r in top3)
    print(f"  b={b:<5}{linea}")


efecto de b sobre 'seguridad' (top 3):

  b=0.0  AMLO    2019-10-15 0.62   AMLO    2023-01-19 0.62   AMLO    2019-10-14 0.62
  b=0.5  AMLO    2023-01-19 0.62   AMLO    2019-10-31 0.61   AMLO    2022-09-09 0.61
  b=0.75 AMLO    2019-10-31 0.62   AMLO    2023-01-19 0.62   Sánchez 2025-03-06 0.61
  b=1.0  AMLO    2019-10-31 0.62   Piñera  2020-01-06 0.62   Piñera  2019-07-08 0.62


## TF-IDF vs BM25 sobre la misma consulta

Mismo corpus y mismas listas de postings; TF-IDF crece lineal con la frecuencia, BM25 la satura.

In [29]:
q = "corrupción"
a, bm = search_tfidf(ix, q, k=5), search_bm25(ix, q, k=5)

print(f"\n  consulta: {q}\n")
print(f"  {'#':<3}{'TF-IDF':<34}{'BM25'}")
for i in range(5):
    izq = f"{SHORT[a[i]['speaker']]:<8}{a[i]['date']}  {a[i]['score']:.3f}"
    der = f"{SHORT[bm[i]['speaker']]:<8}{bm[i]['date']}  {bm[i]['score']:.2f}"
    print(f"  {i + 1:<3}{izq:<34}{der}")


  consulta: corrupción

  #  TF-IDF                            BM25
  1  AMLO    2019-02-13  0.407         Sánchez 2025-06-17  1.90
  2  Sánchez 2025-06-17  0.304         AMLO    2019-02-13  1.89
  3  AMLO    2019-02-07  0.232         Sánchez 2025-07-09  1.88
  4  AMLO    2019-12-27  0.229         AMLO    2019-12-27  1.87
  5  AMLO    2019-09-06  0.219         AMLO    2019-02-07  1.86


## Búsqueda booleana

Gramática: `AND` / `OR` / `NOT` (`&`, `|`, `!`), paréntesis, frases entre comillas y **AND implícito**.

Los términos pasan por el mismo analizador. Dos casos se reportan al margen:

- **stopword** — no está indexada, no filtra
- **fuera de vocabulario** — `df < min_df` o inexistente, no coincide con nada

In [30]:
queries = [
    "presupuesto AND educación",
    "presupuesto educación",
    "milei OR maduro",
    "NOT inflación",
    "(presupuesto AND educación) AND NOT corrupción",
    '"estado de emergencia"',
    "estado AND emergencia",
    "xyzzy AND libertad",
]

print()
for q in queries:
    hits, avisos = parse_boolean(ix, q, diagnostics=True)
    nota = ""
    if avisos["ignored_stopwords"]:
        nota += f"   stopword: {', '.join(avisos['ignored_stopwords'])}"
    if avisos["unknown_terms"]:
        nota += f"   fuera de vocabulario: {', '.join(avisos['unknown_terms'])}"
    print(f"  {q:<52}{hits.size:>5} docs{nota}")


  presupuesto AND educación                            1556 docs
  presupuesto educación                                1556 docs
  milei OR maduro                                       261 docs
  NOT inflación                                        3612 docs
  (presupuesto AND educación) AND NOT corrupción        295 docs
  "estado de emergencia"                                 39 docs
  estado AND emergencia                                1141 docs   stopword: estado
  xyzzy AND libertad                                      0 docs   fuera de vocabulario: xyzzy


In [31]:
show(search_boolean(ix, '"estado de emergencia"', limit=5), 'Booleana · frase exacta')

print()
info({
    "docs con 'inflación'": parse_boolean(ix, "inflación").size,
    "  ... de Javier Milei": parse_boolean(ix, "inflación", speaker="Javier Milei").size,
})


Booleana · frase exacta
  1. AMLO    2024-08-23  23.08.24 Versión estenográfica de la conferencia
     …a nuestro país, preguntarle si van a declarar estado de emergencia en Chalco, si van a hacer… digam
  2. AMLO    2024-06-24  24.06.24 Versión estenográfica de la conferencia
     …derrite al Sistema Eléctrico Nacional, entra en estado de emergencia’. Milenio: ‘Capacidad en gener
  3. AMLO    2022-12-22  22.12.22 Versión estenográfica de la conferencia
     …Unidos en el Perú. Y luego, cuando declaran estado de emergencia, va la embajadora del Perú a entre
  4. AMLO    2022-12-19  19.12.22 Versión estenográfica de la conferencia
     …al presidente. Entonces, pero luego decretan estado de emergencia y sale la policía y sale el ejérc
  5. AMLO    2022-12-16  16.12.22 Versión estenográfica de la conferencia
     …Consejo de Estado se iba a evaluar establecer en Estado de emergencia a todo el país a raíz de las 

  docs con 'inflación'                919
    ... de Javier Milei         

## Índice invertido por dentro

Sobre estas mismas listas operan los dos mecanismos:

- **BM25** acumula scores sobre `postings_indices` / `postings_tf`
- **booleana** aplica `intersect1d` / `union1d` / `setdiff1d` sobre esos arreglos

Los postings están ordenados por `doc_id`, así que las operaciones de conjuntos son lineales.

In [32]:
print("  término        idf     df   tf max   primeros doc_id")
for term in ["inflacion", "corrupcion", "libertad"]:
    j = ix.term_id(term)
    docs, tf = ix.postings(j)
    print(f"  {term:<14}{ix.idf[j]:>4.2f} {int(ix.df[j]):>6d} {int(tf.max()):>7d}   "
          + ", ".join(str(d) for d in docs[:6]))

doc_id = int(ix.docs_of_speaker("Javier Milei")[0])
doc = read_doc(doc_id, ix.corpus_path, ix.offsets)
print()
info({
    "doc_id": doc_id,
    "hablante": SHORT[ix.speakers[doc_id]],
    "fecha": ix.dates[doc_id],
    "título": ix.titles[doc_id][:55],
    "caracteres": f"{len(doc['text']):,}".replace(",", " "),
    "tokens indexados": ix.dl[doc_id],
}, ancho=18)

  término        idf     df   tf max   primeros doc_id
  inflacion     2.59    919      57   1, 3, 9, 10, 11, 21
  corrupcion    1.78   2076      66   0, 1, 2, 3, 4, 5
  libertad      2.67    850       7   12, 14, 15, 55, 72, 81

  doc_id            2 294
  hablante          Milei
  fecha             2023-12-10
  título            Palabras del presidente de la Nación, Javier Milei, lue
  caracteres        1 973
  tokens indexados  167


## `w_global` para operaciones posteriores

La matriz queda persistida y se recarga sin reconstruir nada. Aquí se comprueba que lo guardado es lo que se usa, y se muestran dos operaciones típicas aguas abajo: perfil de un documento y similitud entre documentos.

In [33]:
w_reload = sp.load_npz(config.W_GLOBAL_NPZ).tocsr()
norms = np.sqrt(w_global.multiply(w_global).sum(axis=1)).A1

info({
    "forma": f"{w_global.shape[0]} x {w_global.shape[1]}",
    "entradas": int(w_global.nnz),
    "tipo": str(w_global.dtype),
    "recargada idéntica": bool((w_global != w_reload).nnz == 0),
    "norma L2 de filas": f"{norms.min():.6f} - {norms.max():.6f}",
})


def top_terms(doc_id, n=8):
    row = w_global.getrow(doc_id)
    order = np.argsort(-row.data)[:n]
    return " · ".join(f"{ix.terms[c]} {w:.2f}" for c, w in zip(row.indices[order], row.data[order]))


print("\n  términos de mayor peso en el último discurso de cada hablante:\n")
for speaker in ["Javier Milei", "Pedro Sánchez", "Andrés Manuel López Obrador"]:
    doc_id = int(ix.docs_of_speaker(speaker)[-1])
    print(f"  {SHORT[speaker]:<8}{ix.dates[doc_id]}  {top_terms(doc_id)}")

d1 = int(ix.docs_of_speaker("Javier Milei")[0])
d2 = int(ix.docs_of_speaker("Pedro Sánchez")[0])
print()
info({"coseno Milei[0] vs Sánchez[0]": round(float(w_global[d1].dot(w_global[d2].T).toarray()[0, 0]), 4)})

  forma                               4531 x 30083
  entradas                            3 687 905
  tipo                                float64
  recargada idéntica                  1
  norma L2 de filas                   1.000000 - 1.000000

  términos de mayor peso en el último discurso de cada hablante:

  Milei   2026-04-29  argentin 0.21 · capital 0.19 · model 0.16 · keyn 0.15 · dec 0.14 · hac 0.14 · crecimient 0.14 · per 0.13
  Sánchez 2026-04-29  señori 0.40 · feijo 0.24 · construct 0.22 · vox 0.18 · reflexion 0.18 · señor 0.17 · parlamentari 0.17 · gobiern 0.15
  AMLO    2024-09-30  cient 0.35 · president 0.18 · mil 0.17 · adel 0.16 · sexeni 0.14 · lopez 0.14 · calderon 0.13 · obrador 0.13

  coseno Milei[0] vs Sánchez[0]       0.0545


# Tests

Comprobaciones sobre las tres capas: **corpus**, **índice** y **buscadores**.

`check(...)` acumula en lugar de detener la ejecución, así que se ven todos los fallos de golpe. La última celda imprime el recuento y solo entonces falla con `AssertionError`.

In [34]:
import re
from datetime import date

from ri_scripts.boolean_search import BooleanSyntaxError

RESULTS = []


def check(nombre, condicion, detalle=""):
    RESULTS.append({"prueba": nombre, "ok": bool(condicion), "detalle": str(detalle)[:55]})
    return bool(condicion)


def check_raises(nombre, fn, exc=Exception):
    try:
        fn()
    except exc as e:
        return check(nombre, True, type(e).__name__)
    return check(nombre, False, f"no lanzó {exc.__name__}")


records = list(iter_corpus(config.CORPUS_JSONL))
n = len(records)

src = pd.read_parquet(config.PARQUET)
check("docs == filas del parquet", len(src) == n, f"{len(src)} vs {n}")
check("mismos caracteres de texto",
      int(src.speech_text.fillna("").astype(str).str.len().sum()) == sum(len(r["text"]) for r in records))
check("doc_id consecutivo 0..N-1", [r["doc_id"] for r in records] == list(range(n)))
check("offsets alineados", len(ix.offsets) == n, len(ix.offsets))
check("speech_id único", len({r["speech_id"] for r in records}) == n)
check("campos obligatorios",
      all({"doc_id", "speech_id", "speaker", "country", "date", "title", "text"} <= set(r) for r in records))
check("texto no vacío", all(r["text"].strip() for r in records))
check("títulos no vacíos", all(r["title"].strip() for r in records))
check("fechas ISO", not [r["date"] for r in records if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", r["date"])])
try:
    {date.fromisoformat(r["date"]) for r in records}
    check("fechas parseables", True)
except ValueError as e:
    check("fechas parseables", False, e)
check("países esperados", {r["country"] for r in records} == {"ARG", "CHL", "ESP", "MEX", "VEN"},
      sorted({r["country"] for r in records}))

muestra = [0, 1, 2293, 2294, 2480, 2510, 2511, 3646, 3647, n - 1]
check("read_doc(offset) == lectura secuencial",
      not [i for i in muestra if read_doc(i, config.CORPUS_JSONL, ix.offsets) != records[i]])

bloques = sorted(ix.speaker_blocks.items(), key=lambda par: par[1][0])
check("bloques sin huecos ni solapes",
      bloques[0][1][0] == 0 and bloques[-1][1][1] == n
      and all(a[1][1] == b[1][0] for a, b in zip(bloques, bloques[1:])),
      f"{len(bloques)} bloques")
check("hablante correcto en cada bloque",
      all(all(ix.speakers[i] == spk for i in range(s, e)) for spk, (s, e) in ix.speaker_blocks.items()))
check("fechas ordenadas por hablante",
      all(all(ix.dates[i] <= ix.dates[i + 1] for i in range(s, e - 1)) for s, e in ix.speaker_blocks.values()))

True

### Muestra del corpus

Tres documentos por hablante, leídos con `read_doc` (acceso por offset, no secuencial).

In [35]:
for speaker, (start, end) in ix.speaker_blocks.items():
    print(f"\n  {SHORT[speaker]}   (doc_id {start}-{end - 1})")
    for doc_id in (start, (start + end) // 2, end - 1):
        doc = read_doc(doc_id, ix.corpus_path, ix.offsets)
        print(f"    {doc_id:<5}{doc['date']}  {int(ix.dl[doc_id]):>5} tok  {doc['title'][:40]}")
        print(f"           \"{' '.join(doc['text'].split())[:80]}\"")


  AMLO   (doc_id 0-2293)
    0    2018-12-02    938 tok  Discurso del presidente Andrés Manuel Ló
           "~~~PRESIDENTE ANDRÉS MANUEL LÓPEZ OBRADORDescarga discurso: AMLO discurso FArmad"
    1147 2021-05-24   5072 tok  24.05.21 Versión estenográfica de la con
           "~~~PRESIDENTE ANDRÉS MANUEL LÓPEZ OBRADOR Buenos días. Bueno, comenzamos la sema"
    2293 2024-09-30   2635 tok  30.09.24 Versión estenográfica de la últ
           "~~~PRESIDENTE ANDRÉS MANUEL LÓPEZ OBRADOR Buenos días. Ánimo, ánimo. Bueno, pues"

  Milei   (doc_id 2294-2480)
    2294 2023-12-10    167 tok  Palabras del presidente de la Nación, Ja
           "Hola a todos. ¡Viva la libertad, carajo! ¡Viva la libertad, carajo! ¡Viva la lib"
    2387 2025-01-19    663 tok  Palabras del Presidente Javier Milei en 
           "Buenas noches a todos. Quiero agradecer a las autoridades de Latino Wall Street "
    2480 2026-04-29   4857 tok  Discurso del Presidente Javier Milei en 
           "Javier Milei: Buenas tar

### Preproceso e índice

El analizador es el mismo al indexar y al consultar; los tres artefactos (`postings`, `df`/`idf`, `w_global`) describen la misma matriz.

In [36]:
from ri_scripts.preprocess import analyze_phrase, fold, stem

check("fold conserva la longitud (mapea posiciones)",
      all(len(fold(s)) == len(s) for s in ["México, año 2024", "ÁÉÍÓÚ ñ", "¿Qué tal?"]))
check("acentos plegados", fold("inflación") == "inflacion", fold("inflación"))
check("ñ conservada", fold("año") == "año", fold("año"))
check("mayúsculas normalizadas", fold("MÉXICO") == "mexico")

t = analyze("el pueblo de México y la gente, sí")
check("stopwords eliminadas", not (set(t) & STOPWORDS), sorted(set(t) & STOPWORDS))
check("contenido conservado", {"puebl", "mexic"} <= set(t), t)
check("consulta solo con stopwords", analyze("de la y el") == [])
check("stemming agrupa flexión", len({stem("economía"), stem("económico"), stem("economías")}) == 1)
check("frases conservan stopwords", "de" in analyze_phrase("estado de emergencia"))

check("|indptr| == V+1", len(ix.postings_indptr) == ix.vocab_size + 1)
check("indptr no decreciente", bool(np.all(np.diff(ix.postings_indptr) >= 0)))
check("nnz(indices) == nnz(tf) == nnz(w_global)",
      len(ix.postings_indices) == len(ix.postings_tf) == w_global.nnz)
check("doc_id dentro de rango", int(ix.postings_indices.max()) < ix.n_docs)
check("postings ordenados por doc_id",
      all(np.all(np.diff(ix.postings_indices[s:e]) > 0)
          for s, e in zip(ix.postings_indptr[:-1], ix.postings_indptr[1:]) if e > s))
check("df == longitud de los postings", np.array_equal(ix.df, np.diff(ix.postings_indptr)))
check("idf == ln((1+N)/(1+df)) + 1", np.allclose(ix.idf, np.log((1 + ix.n_docs) / (1 + ix.df)) + 1))
check("idf decreciente con df", bool(np.all(np.diff(ix.idf[np.argsort(ix.df)]) <= 1e-12)))
check("vocabulario <-> términos", all(ix.vocabulary[term] == j for j, term in enumerate(ix.terms)))
check("ids de término cubren 0..V-1", set(ix.vocabulary.values()) == set(range(ix.vocab_size)))

norms = np.sqrt(w_global.multiply(w_global).sum(axis=1)).A1
check("filas de w_global con norma 1", np.allclose(norms, 1.0, atol=1e-9),
      f"{norms.min():.6f}-{norms.max():.6f}")
check("tokens >= términos distintos por doc", bool(np.all(ix.dl >= np.diff(w_global.indptr))))
check("avgdl == media de dl", np.isclose(ix.avgdl, ix.dl.mean()))

rng = np.random.default_rng(0)
proporcional = True
for r in rng.choice(ix.n_docs, 8, replace=False):
    row = w_global.getrow(int(r))
    ratios = []
    for col, val in zip(row.indices, row.data):
        docs, tf = ix.postings(int(col))
        ratios.append(val / (tf[np.searchsorted(docs, r)] * ix.idf[col]))
    proporcional &= bool(np.allclose(ratios, ratios[0], rtol=1e-9))
check("w_global proporcional a tf·idf por fila", proporcional)
check("w_global guardada == en memoria", (w_global != sp.load_npz(config.W_GLOBAL_NPZ).tocsr()).nnz == 0)

True

### Buscadores

Ranking coherente, semántica correcta de los operadores y filtrado por hablante.

In [37]:
res = search_tfidf(ix, "inflación", k=20)
docs_infl = set(ix.postings(ix.term_id("inflacion"))[0].tolist())
check("tfidf devuelve k resultados", len(res) == 20, len(res))
check("tfidf ordenado por score", all(a["score"] >= b["score"] - 1e-12 for a, b in zip(res, res[1:])))
check("tfidf scores en (0, 1]", all(0.0 < r["score"] <= 1.0 + 1e-9 for r in res))
check("tfidf solo docs con el término", {r["doc_id"] for r in res} <= docs_infl)
check("tfidf solo stopwords", search_tfidf(ix, "de la y el", k=5) == [])
check("tfidf término inexistente", search_tfidf(ix, "xyzzy", k=5) == [])
check("tfidf filtro por hablante",
      {r["speaker"] for r in search_tfidf(ix, "inflación", k=10, speaker="Javier Milei")} == {"Javier Milei"})

res_b = search_bm25(ix, "inflación", k=20)
check("bm25 devuelve k resultados", len(res_b) == 20, len(res_b))
check("bm25 ordenado por score", all(a["score"] >= b["score"] - 1e-12 for a, b in zip(res_b, res_b[1:])))
check("bm25 scores positivos", all(r["score"] > 0 for r in res_b))
check("bm25 solo docs con el término", {r["doc_id"] for r in res_b} <= docs_infl)
check("bm25 término inexistente", search_bm25(ix, "xyzzy", k=5) == [])

docs_p, tf_p = ix.postings(ix.term_id("inflacion"))
score_b0 = {r["doc_id"]: r["score"] for r in search_bm25(ix, "inflación", k=ix.n_docs, b=0.0)}
pares = sorted(zip(tf_p.tolist(), [score_b0[d] for d in docs_p.tolist()]))
check("bm25 monótono en tf (b=0)", all(a[1] <= b[1] + 1e-9 for a, b in zip(pares, pares[1:])))

a = set(parse_boolean(ix, "presupuesto").tolist())
b = set(parse_boolean(ix, "educación").tolist())
check("AND = intersección", set(parse_boolean(ix, "presupuesto AND educación").tolist()) == a & b)
check("AND implícito == explícito",
      set(parse_boolean(ix, "presupuesto educación").tolist())
      == set(parse_boolean(ix, "presupuesto AND educación").tolist()))
check("OR = unión", set(parse_boolean(ix, "presupuesto OR educación").tolist()) == a | b)
check("NOT = complemento", set(parse_boolean(ix, "NOT presupuesto").tolist()) == set(range(ix.n_docs)) - a)
check("A AND NOT A = vacío", parse_boolean(ix, "presupuesto AND NOT presupuesto").size == 0)
check("idempotencia A OR A", set(parse_boolean(ix, "presupuesto OR presupuesto").tolist()) == a)
check("precedencia A OR B AND C == A OR (B AND C)",
      set(parse_boolean(ix, "presupuesto OR educación AND corrupción").tolist())
      == set(parse_boolean(ix, "presupuesto OR (educación AND corrupción)").tolist()))

frase = set(parse_boolean(ix, '"estado de emergencia"').tolist())
check("frase contenida en el AND de sus términos",
      frase <= set(parse_boolean(ix, "estado AND emergencia").tolist()))
check("stopword no filtra", parse_boolean(ix, "estado").size == ix.n_docs)
check("fuera de vocabulario devuelve vacío", parse_boolean(ix, "xyzzy").size == 0)
check("filtro por hablante en booleana",
      set(parse_boolean(ix, "inflación", speaker="Javier Milei").tolist())
      <= set(ix.docs_of_speaker("Javier Milei").tolist()))
check_raises("paréntesis sin cerrar", lambda: parse_boolean(ix, "(presupuesto AND educación"), BooleanSyntaxError)
check_raises("operador sin operando", lambda: parse_boolean(ix, "presupuesto AND"), BooleanSyntaxError)
check_raises("paréntesis de más", lambda: parse_boolean(ix, "presupuesto)"), BooleanSyntaxError)
check_raises("hablante inexistente", lambda: search_tfidf(ix, "inflación", speaker="Nadie"), KeyError)

True

In [38]:
res = pd.DataFrame(RESULTS)
fallos = res[~res.ok]

print(f"{len(res) - len(fallos)}/{len(res)} pruebas OK")
for _, r in fallos.iterrows():
    print(f"  x  {r['prueba']}   {r['detalle']}")

if len(fallos):
    raise AssertionError(f"{len(fallos)} pruebas fallando")

67/67 pruebas OK
